In [0]:
# Read the data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, current_timestamp

##
## CSV -> Bronze
##

# Set the path to the CSV file in the DEV volume
csv_path = f'/Volumes/workspace/default/test_volume/test_directory/health/'


# Define the schema for the CSV file
health_csv_schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("PII", StringType(), True),
    StructField("date", DateType(), True),
    StructField("HighCholest", IntegerType(), True),
    StructField("HighBP", DoubleType(), True),
    StructField("BMI", DoubleType(), True),
    StructField("Age", DoubleType(), True),
    StructField("Education", DoubleType(), True),
    StructField("income", IntegerType(), True)
])

# Ingest the CSV file and add metadata columns for the ingested data
health_raw = (
    spark
    .read
    .format("csv") 
    .option("header", "true")             # Use the header row for column names
    .schema(health_csv_schema)            # Apply the defined schema
    .load(csv_path)                       # Load the CSV data
    .select(
        "*",
        "_metadata.file_name",                        # Include file name from metadata
        "_metadata.file_modification_time",           # Include file modification timestamp
        current_timestamp().alias("processing_time")  # Add a processing time column
    )
)


# Save the ingested data as a Bronze table in Delta format
(health_raw
 .write
 .format("delta")
 .mode('overwrite')  # Overwrite existing data
 .saveAsTable(f"workspace.default.health_bronze_dev")
)

health_bronze = spark.table(f'workspace.default.health_bronze_dev')


##
## Bronze -> Silver
##

health_silver = (
    health_bronze
    # Create a new column to categorize the HighCholest column
    .withColumn(
        "HighCholest_Group", 
        when(col("HighCholest") == 0, 'Normal')
        .when(col("HighCholest") == 1, 'Above Average')
        .when(col("HighCholest") == 2, 'High')
        .otherwise('Unknown')
    )
    # Create a new column to categorize the Age_Group column
    .withColumn(
        "Age_Group", 
        when(col("Age") <= 9, "0-9")
        .when((col("Age") >= 10) & (col("Age") <= 19), "10-19")
        .when((col("Age") >= 20) & (col("Age") <= 29), "20-29")
        .when((col("Age") >= 30) & (col("Age") <= 39), "30-39")
        .when((col("Age") >= 40) & (col("Age") <= 49), "40-49")
        .when(col("Age") >= 50, "50+")
        .otherwise('Unknown')
    )
    # Drop unnecessary columns (e.g., metadata columns)
    .drop("file_name", "file_modification_time", "processing_time")
)

# Save the transformed data as a Silver table
(health_silver
 .write
 .format("delta")
 .mode("overwrite")  # Overwrite any existing data
 .saveAsTable(f"workspace.default.health_silver_dev")
)



##
## Silver - Gold
##
# Create the summarization view
chol_age_agg = spark.sql(f'''
    CREATE OR REPLACE TABLE workspace.default.chol_age_agg_dev AS
    SELECT 
        HighCholest_Group, 
        Age_Group, 
        count(*) as Total
    FROM workspace.default.health_silver_dev
    GROUP BY HighCholest_Group, Age_Group
''')

## Display the final gold table
spark.table(f'workspace.default.chol_age_agg_dev').display()


### Modularize the pyspark code above

Modularize the code from above into the following functions:
 
  a. `get_health_csv_schema`: Defines and returns the schema for the CSV files.

  b. `read_health_data`: Reads the CSV file into a DataFrame with metadata.

  c. `high_cholest_map` and `group_ages_map`: Converts the numeric values to categorical. 

  d. `save_df_to_delta`: Saves the transformed data to a Delta table.

  e. `get_cholest_age_agg`: Aggregates the data for the gold table.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, current_timestamp


##
## Ingest Cloud to Bronze
##

# a. Function to define the schema for the health data CSV
def get_health_csv_schema():
    return StructType([
        StructField("ID", IntegerType(), True),
        StructField("PII", StringType(), True),
        StructField("date", DateType(), True),
        StructField("HighCholest", IntegerType(), True),
        StructField("HighBP", DoubleType(), True),
        StructField("BMI", DoubleType(), True),
        StructField("Age", DoubleType(), True),
        StructField("Education", DoubleType(), True),
        StructField("income", IntegerType(), True)
    ])

# b. Function to read the CSV data into a DataFrame and add metadata columns
def read_health_data(csv_path, schema):
    return (
        spark
        .read
        .format("csv")
        .option("header", "true")  # Use the header row for column names
        .schema(schema)            # Apply the defined schema
        .load(csv_path)            # Load the CSV data
        .select(
            "*",
            "_metadata.file_name",                        # Include file name from metadata
            "_metadata.file_modification_time",           # Include file modification timestamp
            current_timestamp().alias("processing_time")  # Add a processing time column
        )
    )

##
## Mapping Functions for Data Transformation
##

# c. Map the 'HighCholest' column to categories
def high_cholest_map(col_name):
    return (
        when(col(col_name) == 0, 'Normal')
        .when(col(col_name) == 1, 'Above Average')
        .when(col(col_name) == 2, 'High')
        .otherwise('Unknown')
    )


# d. Map the 'Age' column to age groups
def group_ages_map(col_name):
    return (
        when(col(col_name) <= 9, "0-9")
        .when((col(col_name) >= 10) & (col(col_name) <= 19), "10-19")
        .when((col(col_name) >= 20) & (col(col_name) <= 29), "20-29")
        .when((col(col_name) >= 30) & (col(col_name) <= 39), "30-39")
        .when((col(col_name) >= 40) & (col(col_name) <= 49), "40-49")
        .when(col(col_name) >= 50, "50+")
        .otherwise('Unknown')
    )

##
## Save a DataFrame to Delta Table
##

# e. Function to save the DataFrame to a Delta table
def save_df_to_delta(dataframe, uc_table, mode):
    (dataframe
     .write
     .format("delta")
     .mode(mode)             # Specify the save mode (e.g., 'overwrite', 'append')
     .saveAsTable(uc_table)  # Save the DataFrame as a table
    )


##
## Gold Aggregation
##

# f. Function to create a Gold-level table with aggregated counts
def get_cholest_age_agg(catalog, schema, table_name):
    query = f'''
        CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} AS
        SELECT 
            HighCholest_Group, 
            Age_Group, 
            count(*) as Total
        FROM {catalog}.{schema}.health_silver_dev
        GROUP BY HighCholest_Group, Age_Group
    '''
    return spark.sql(query)

Now, use the defined functions to re-create the previous block of code

In [0]:
##
## CSV to Bronze
##

# Read the health CSV data into a DataFrame and save it to the Bronze table
health_csv_df = read_health_data(
    csv_path = f"/Volumes/workspace/default/test_volume/test_directory/health/", 
    schema = get_health_csv_schema()
)

# Save the raw data as a Bronze table in Delta format
save_df_to_delta(health_csv_df, f"workspace.default.health_bronze_dev", mode="overwrite")


##
## Bronze to Silver
##

# Transform the data by adding new columns and cleaning up metadata
health_bronze = spark.table(f'workspace.default.health_bronze_dev')

silver_df = (
    health_bronze
    .withColumn("HighCholest_Group", high_cholest_map("HighCholest"))    # Categorize HighCholest
    .withColumn("Age_Group", group_ages_map("Age"))                     # Categorize Age
    .drop("file_name", "file_modification_time", "processing_time")     # Drop unnecessary metadata columns
)

# Save the transformed data as a Silver table in Delta format
save_df_to_delta(silver_df, f"workspace.default.health_silver_dev", mode="overwrite")


##
## Gold Table
##

# Aggregate the data at the Gold level by cholesterol and age groups
get_cholest_age_agg(catalog = 'workspace', schema = 'default', table_name ='chol_age_agg_dev')

In [0]:
spark.table("workspace.default.health_bronze_dev").display()

In [0]:
spark.table("workspace.default.health_silver_dev").display()

In [0]:
spark.table("workspace.default.chol_age_agg_dev").display()

In [0]:
# clean the table
def clean_table(catalog):
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.default.health_bronze_dev")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.default.health_silver_dev")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.default.chol_age_agg_dev")

clean_table("workspace")

## Unit Test

In [0]:
# import the helper function (which is just the copy of the functions created in the modularization section)
import helper_func # since the helper function is in the same folder as this notebook, we can just import it


#### Test Schema
assertSchemaEqual()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, LongType
from pyspark.sql.functions import col, when

from pyspark.testing.utils import assertSchemaEqual # we will unit test one function from the helpful functions 

def test_get_health_csv_schema():
    # get the schema from our function
    actual_schema = helper_func.get_health_csv_schema()

    # defin the expected schema that the helper function should return
    expected_schema = StructType([
        StructField("ID", IntegerType(), True),
        StructField("PII", StringType(), True),
        StructField("date", DateType(), True),
        StructField("HighCholest", IntegerType(), True),
        StructField("HighBP", DoubleType(), True),
        StructField("BMI", DoubleType(), True),
        StructField("Age", DoubleType(), True),
        StructField("Education", DoubleType(), True),
        StructField("income", IntegerType(), True)
    ])

    # assert the actual schema matches the expected schema
    assertSchemaEqual(actual_schema, expected_schema)
    print("Schema Match! Test Pass")

test_get_health_csv_schema()

#### Test dataframe
assertDataFrameEqual()

In [0]:
# import assertDataFrameEqual
from pyspark.testing.utils import assertDataFrameEqual

def test_high_cholest_map():
  # create the sample dataframe to test
  df = spark.createDataFrame(
      [(0,),
        (1,),
        (2,),
        (3,),
        (4,),
        (None,)],
      ['value']
  )

  # apply the func on the sample data
  actual_df = df.withColumn("actual", helper_func.high_cholest_map("value"))

  # create the static expected dataframe
  expected_df = spark.createDataFrame(
    [
      (0,'Normal'),
      (1,'Above Average'),
      (2,'High'),
      (3,'Unknown'),
      (4,'Unknown'),
      (None, 'Unknown')
    ],
    schema = StructType([
      StructField("value", LongType(), True),
      StructField("actual", StringType(), True)
    ])
  )

  # assert the actual dataframe matches the expected dataframe
  assertDataFrameEqual(actual_df, expected_df)
  print("Test Pass")

test_high_cholest_map()

## Create a File for Unit Test to automate the testing with Pytest

In [0]:
!pip install pytest==8.3.4

In [0]:
import pytest, sys

sys.dont_write_bytecode = True

retcode = pytest.main(["test_helper_func.py", "-v", "-p","no:cacheprovider"]) # if we do not provide the specific file, pytest will scan the current folder and run all the test files with 'test_' prefix

# faill the cell if any test fails
assert retcode == 0, "pytest failed"